# DBSCAN Clustering Demonstration

This notebook demonstrates the DBSCAN clustering algorithm implemented in Rust using various scikit-learn datasets.

## Overview

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) is a density-based clustering algorithm that:
- Groups together points that are closely packed together
- Marks points in low-density regions as outliers (noise)
- Does not require specifying the number of clusters beforehand

**Parameters:**
- `eps`: Maximum distance between two samples for one to be considered in the neighborhood of the other
- `min_samples`: The number of samples in a neighborhood for a point to be considered as a core point

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from ghdbscan import DBSCAN
import time

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Helper Functions

In [ ]:
def plot_clusters(X, labels, title, ax=None):
    """Plot clustering results."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    unique_labels = set(labels)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Noise points
            col = 'gray'
            marker = 'x'
            alpha = 0.3
            label = 'Noise'
        else:
            marker = 'o'
            alpha = 0.7
            label = f'Cluster {k}'
        
        class_member_mask = (labels == k)
        xy = X[class_member_mask]
        ax.scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, 
                  s=50, alpha=alpha, edgecolors='k', linewidth=0.5, label=label)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Feature 1', fontsize=12)
    ax.set_ylabel('Feature 2', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    return ax

def print_clustering_stats(labels, elapsed_time=None):
    """Print clustering statistics."""
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    print(f"Number of clusters: {n_clusters}")
    print(f"Number of noise points: {n_noise}")
    if elapsed_time:
        print(f"Clustering time: {elapsed_time:.4f} seconds")
    print()

## 1. Isotropic Gaussian Blobs

Testing DBSCAN on well-separated Gaussian clusters.

In [ ]:
# Generate sample data
X_blobs, y_blobs = datasets.make_blobs(n_samples=300, centers=4, 
                                        cluster_std=0.6, random_state=42)

# Run DBSCAN
print("DBSCAN on Gaussian Blobs")
print("=" * 50)
dbscan = DBSCAN(eps=0.8, min_samples=5)

start_time = time.time()
labels = dbscan.fit_predict(X_blobs)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data (Ground Truth)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.grid(True, alpha=0.3)

# DBSCAN results
plot_clusters(X_blobs, labels, 'DBSCAN Results (eps=0.8, min_samples=5)', ax=ax2)

plt.tight_layout()
plt.show()

## 2. Moons Dataset

Testing DBSCAN on non-convex clusters (interleaving half circles).

In [ ]:
# Generate moons dataset
X_moons, y_moons = datasets.make_moons(n_samples=300, noise=0.05, random_state=42)

# Run DBSCAN
print("DBSCAN on Moons Dataset")
print("=" * 50)
dbscan = DBSCAN(eps=0.2, min_samples=5)

start_time = time.time()
labels = dbscan.fit_predict(X_moons)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data (Ground Truth)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.grid(True, alpha=0.3)

# DBSCAN results
plot_clusters(X_moons, labels, 'DBSCAN Results (eps=0.2, min_samples=5)', ax=ax2)

plt.tight_layout()
plt.show()

## 3. Circles Dataset

Testing DBSCAN on concentric circles.

In [ ]:
# Generate circles dataset
X_circles, y_circles = datasets.make_circles(n_samples=300, factor=0.5, 
                                              noise=0.05, random_state=42)

# Run DBSCAN
print("DBSCAN on Circles Dataset")
print("=" * 50)
dbscan = DBSCAN(eps=0.15, min_samples=5)

start_time = time.time()
labels = dbscan.fit_predict(X_circles)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data (Ground Truth)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.grid(True, alpha=0.3)

# DBSCAN results
plot_clusters(X_circles, labels, 'DBSCAN Results (eps=0.15, min_samples=5)', ax=ax2)

plt.tight_layout()
plt.show()

## 4. Iris Dataset

Testing DBSCAN on the classic Iris dataset (using first 2 features for visualization).

In [ ]:
# Load Iris dataset
iris = datasets.load_iris()
X_iris = iris.data[:, :2]  # Use first 2 features for visualization
y_iris = iris.target

# Standardize features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Run DBSCAN
print("DBSCAN on Iris Dataset (first 2 features)")
print("=" * 50)
dbscan = DBSCAN(eps=0.5, min_samples=5)

start_time = time.time()
labels = dbscan.fit_predict(X_iris_scaled)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1], c=y_iris, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data (Ground Truth)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sepal Length (scaled)', fontsize=12)
ax1.set_ylabel('Sepal Width (scaled)', fontsize=12)
ax1.grid(True, alpha=0.3)

# DBSCAN results
plot_clusters(X_iris_scaled, labels, 'DBSCAN Results (eps=0.5, min_samples=5)', ax=ax2)
ax2.set_xlabel('Sepal Length (scaled)', fontsize=12)
ax2.set_ylabel('Sepal Width (scaled)', fontsize=12)

plt.tight_layout()
plt.show()

## 5. Noisy Data with Outliers

Testing DBSCAN's ability to identify outliers in noisy data.

In [ ]:
# Generate data with outliers
X_noisy, y_noisy = datasets.make_blobs(n_samples=250, centers=3, 
                                        cluster_std=0.5, random_state=42)

# Add random outliers
outliers = np.random.uniform(low=-10, high=10, size=(50, 2))
X_with_outliers = np.vstack([X_noisy, outliers])

# Run DBSCAN
print("DBSCAN on Noisy Data with Outliers")
print("=" * 50)
dbscan = DBSCAN(eps=0.8, min_samples=5)

start_time = time.time()
labels = dbscan.fit_predict(X_with_outliers)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
plot_clusters(X_with_outliers, labels, 
             'DBSCAN on Noisy Data (eps=0.8, min_samples=5)', ax=ax)
plt.show()

## 6. Parameter Sensitivity Analysis

Exploring how different `eps` values affect clustering results.

In [ ]:
# Use moons dataset for parameter analysis
X_test, _ = datasets.make_moons(n_samples=300, noise=0.05, random_state=42)

eps_values = [0.1, 0.2, 0.3, 0.5]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, eps in enumerate(eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_test)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    title = f'eps={eps}, min_samples=5\n({n_clusters} clusters, {n_noise} noise points)'
    plot_clusters(X_test, labels, title, ax=axes[idx])

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the DBSCAN clustering algorithm on various datasets:

1. **Gaussian Blobs**: Successfully identified well-separated clusters
2. **Moons**: Handled non-convex cluster shapes effectively
3. **Circles**: Identified concentric circular patterns
4. **Iris**: Clustered real-world botanical data
5. **Noisy Data**: Effectively identified and separated outliers
6. **Parameter Sensitivity**: Showed how `eps` affects clustering results

### Key Observations:

- DBSCAN excels at finding clusters of arbitrary shape
- It automatically identifies outliers (noise points)
- The `eps` parameter significantly affects results and should be tuned based on data characteristics
- The Rust implementation provides fast clustering performance